In [8]:
import sys
sys.path.insert(0, '..')
from support.paths import resolve
import os
import pandas as pd
from joblib import Parallel, delayed

In [9]:
%run ../0_Config/0_variables.ipynb

In [10]:
X_train = pd.read_parquet("Data/X_train.parquet")
X_validate = pd.read_parquet("Data/X_validate.parquet")
X_test = pd.read_parquet("Data/X_test.parquet")
y_train = pd.read_parquet("Data/y_train.parquet")
y_validate = pd.read_parquet("Data/y_validate.parquet")
y_test = pd.read_parquet("Data/y_test.parquet")

In [11]:
    # Exponential recency weights: newest row ~4.5x oldest (raised from 2x).
    # Stronger recency proved beneficial for NEM markets with regime shifts
    # (Uniejewski et al. 2019: forecasters that up-weight recent observations
    # consistently outperform equal-weight baselines on electricity prices).
    # _recency_w = np.exp(np.linspace(0.0, 1.5, len(train_df))).astype(np.float32)

In [12]:
# Global p97 clip threshold for base model (avoids per-horizon recomputation).

# _clip_thresh = float(np.percentile(aux_vars, BASE_CLIP_PERCENTILE)) if len(aux_vars) > 0 else 300.0

In [13]:
# 1. Base models
# Sample weights: recency × moderate spike-target upweighting.
# base_spike_w = (_recency_w * np.where(y_train[horizon] > SPIKE_THRESHOLD, 3.0, 1.0)).astype(np.float32)

# 1a. L1 base (MAE-optimal, clipped target)
# y_tr_base = _to_asinh(np.minimum(y_train[horizon], _clip_thresh)).astype(np.float32)
# y_va_base = _to_asinh(np.minimum(y_va_raw, _clip_thresh)).astype(np.float32)

# # 1b. L2 base (MSE-optimal, uncapped target) Blending L1+L2 per Lago et al. (2021) reduces both MAE and RMSE.
# y_tr_full_b = _to_asinh(y_train[horizon]).astype(np.float32)
# y_va_full_b = _to_asinh(y_va_raw).astype(np.float32)

In [14]:
def train_seq2seq():



    horizon_list = list(range(1, (int(os.environ["HORIZON_HOURS"]) + 1)))

    def _fit_one(horizon: int):


        

        # 1. Base models 

        # base_m = lgb.LGBMRegressor(**LGBM_BASE_PARAMS)
        # base_m.fit(
        #     X_h_tr, y_tr_base,
        #     sample_weight=base_spike_w,
        #     eval_set=[(X_h_va, y_va_base)],
        #     callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        # )

 


        # base_l2_m = lgb.LGBMRegressor(**LGBM_BASE_L2_PARAMS)
        # base_l2_m.fit(
        #     X_h_tr, y_tr_full_b,
        #     sample_weight=base_spike_w,
        #     eval_set=[(X_h_va, y_va_full_b)],
        #     callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        # )


        # 2. Spike classifier

        # spike_labels_tr = (y_train[target_horizon] > SPIKE_THRESHOLD).astype(np.float32)
        # n_spikes_tr     = int(spike_labels_tr.sum())
        # spike_clf = None
        # spike_reg = None
        # spike_qreg = None
        # dip_clf = None
        # dip_reg = None

        # if n_spikes_tr >= _MIN_SPIKE_TRAIN:
        #     spike_labels_va = (y_va_raw > SPIKE_THRESHOLD).astype(np.float32)


        #     clf = lgb.LGBMClassifier(**LGBM_CLF_PARAMS)
        #     clf.fit(
        #         X_h_tr, spike_labels_tr,
        #         sample_weight=_recency_w,
        #         eval_set=[(X_h_va, spike_labels_va)],
        #         callbacks=[lgb.early_stopping(SPIKE_ES_ROUNDS, verbose=False)],
        #     )
        #     spike_clf = clf





        #     # ── 3. Spike regressor (full range, upweighted spike rows) ────────
        #     y_tr_full = _to_asinh(y_train[horizon]).astype(np.float32)
        #     y_va_full = _to_asinh(y_va_raw).astype(np.float32)
        #     spike_w   = _recency_w * np.where(spike_labels_tr > 0, _SPIKE_UPWEIGHT, 1.0)

        #     sreg = lgb.LGBMRegressor(**LGBM_SPIKE_PARAMS)
        #     sreg.fit(
        #         X_h_tr, y_tr_full,
        #         sample_weight=spike_w,
        #         eval_set=[(X_h_va, y_va_full)],
        #         callbacks=[lgb.early_stopping(SPIKE_ES_ROUNDS, verbose=False)],
        #     )
        #     spike_reg = sreg

        #     qreg = lgb.LGBMRegressor(**LGBM_SPIKE_Q_PARAMS)
        #     qreg.fit(
        #         X_h_tr, y_tr_full,
        #         sample_weight=spike_w,
        #         eval_set=[(X_h_va, y_va_full)],
        #         callbacks=[lgb.early_stopping(SPIKE_ES_ROUNDS, verbose=False)],
        #     )
        #     spike_qreg = qreg






        #     # ── 4. Dip classifier + regressor (negative/low prices) ─────────
        #     dip_labels_tr = (y_train[horizon] < _DIP_THRESHOLD).astype(np.float32)
        #     n_dips_tr     = int(dip_labels_tr.sum())
        #     if n_dips_tr >= _MIN_SPIKE_TRAIN:
        #         dip_labels_va = (y_va_raw < _DIP_THRESHOLD).astype(np.float32)
        #         dclf = lgb.LGBMClassifier(**LGBM_DIP_CLF_PARAMS)
        #         dclf.fit(
        #             X_h_tr, dip_labels_tr,
        #             sample_weight=_recency_w,
        #             eval_set=[(X_h_va, dip_labels_va)],
        #             callbacks=[lgb.early_stopping(SPIKE_ES_ROUNDS, verbose=False)],
        #         )
        #         dip_clf = dclf

        #         dip_w = _recency_w * np.where(dip_labels_tr > 0, _DIP_UPWEIGHT, 1.0)
        #         dreg = lgb.LGBMRegressor(**LGBM_DIP_PARAMS)
        #         dreg.fit(
        #             X_h_tr, y_tr_full,
        #             sample_weight=dip_w,
        #             eval_set=[(X_h_va, y_va_full)],
        #             callbacks=[lgb.early_stopping(SPIKE_ES_ROUNDS, verbose=False)],
        #         )
        #         dip_reg = dreg


        # base_iter = getattr(base_m, "best_iteration_", "?")
        # l2_iter   = getattr(base_l2_m, "best_iteration_", "?")
 

        # base_m = None
        # base_l2_m = None
        # spike_clf = None
        # spike_reg = None
        # spike_qreg = None
        # dip_clf = None
        # dip_reg = None

        return None
    

    # calls the above _fit_one many times; Parallel preserves input order
    trained_models = Parallel(n_jobs=-1, prefer="threads")(
        delayed(_fit_one)(h) for h in horizon_list
    )

# base_models, base_l2_models, spike_clfs, spike_regs, spike_qregs, dip_clfs, dip_regs = map(list, zip(*trained_models))

   

In [15]:
model = train_seq2seq()